# CS 521 HW4, Problem 2: Certified Training for Multiple Norms (CURE)

Open this notebook in Google Colab and select a GPU runtime (*Runtime > Change runtime type > T4 GPU*).

Run the cells in order to train and evaluate the baselines, proposal and ablation.
Do not change the data, the model, the attacks, the verifier, the evaluation or the default hyperparameters.
Change them only if the homework tells you to.

**Threat model.** Let $f$ be the network and $x$ an image with the label $y$. The network is *certified* at $x$
for the norm $p$ if the verifier proves this: $f_y(x') - f_j(x') > 0$ for all $j \neq y$ and all $x'$ with
$\|x' - x\|_p \leq \epsilon_p$. For $\ell_\infty$, $x'$ must also be in $[0, 1]^{784}$.
We use $\epsilon_\infty = 0.1$ and $\epsilon_2 = 1.0$.
The *union certified accuracy* is the percentage of test images where the network is certified for the two norms.

In [1]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cpu":
    torch.set_num_threads(2)
print("device:", device)

EPS = {"linf": 0.1, "l2": 1.0}     # the radii of the threat model
NORMS = ["linf", "l2"]
from pathlib import Path
import json
import math

device: cpu


# Data and model

The data and the network are the same as in Problem 1. The network has 3 hidden layers with 50 neurons.
It has no normalization layer.

In [2]:
def load_mnist(train):
    cache = Path("../../hw2/q4/mnist_data")
    root = cache if (cache / "MNIST/raw").is_dir() else Path("data")
    ds = torchvision.datasets.MNIST(root, train=train, download=True)
    x = ds.data.float().div(255).unsqueeze(1)  # (n, 1, 28, 28)
    return x.to(device), ds.targets.to(device)


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for i in range(0, len(x), batch_size):
        j = idx[i:i + batch_size]
        yield x[j], y[j]


x_train, y_train = load_mnist(train=True)
x_test, y_test = load_mnist(train=False)
print(x_train.shape, x_test.shape)


def make_model():
    return nn.Sequential(
        nn.Flatten(),
        nn.Linear(784, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 50), nn.ReLU(),
        nn.Linear(50, 10),
    ).to(device)


def linear_layers(model):
    return [m for m in model if isinstance(m, nn.Linear)]

torch.Size([60000, 1, 28, 28]) torch.Size([10000, 1, 28, 28])


# Attacks

`pgd(model, x, y, eps, norm, steps)` is an untargeted PGD attack with a random start.
Use `norm="linf"` or `norm="l2"`. The function returns the adversarial images of the restart with the highest loss.

In [3]:
def _project(delta, eps, norm):
    if norm == "linf":
        return delta.clamp(-eps, eps)
    n = delta.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
    return delta * (eps / n).clamp(max=1.0)


def pgd(model, x, y, eps, norm, steps=10, alpha=None, restarts=1):
    alpha = alpha or 2.5 * eps / steps
    best_x, best_loss = x.clone(), torch.full((len(x),), -float("inf"), device=x.device)
    for _ in range(restarts):
        if norm == "linf":
            delta = (torch.rand_like(x) * 2 - 1) * eps
        else:
            delta = torch.randn_like(x)
            r = torch.rand(len(x), 1, 1, 1, device=x.device) * eps
            delta = delta / delta.flatten(1).norm(dim=1).view(-1, 1, 1, 1) * r
        delta = (x + delta).clamp(0, 1) - x
        for _ in range(steps):
            delta.requires_grad_(True)
            loss = F.cross_entropy(model(x + delta), y)
            grad, = torch.autograd.grad(loss, delta)
            with torch.no_grad():
                if norm == "linf":
                    step = grad.sign()
                else:
                    step = grad / grad.flatten(1).norm(dim=1).clamp(min=1e-12).view(-1, 1, 1, 1)
                delta = _project(delta + alpha * step, eps, norm)
                delta = (x + delta).clamp(0, 1) - x
        with torch.no_grad():
            loss = F.cross_entropy(model(x + delta), y, reduction="none")
            better = loss > best_loss
            best_x[better], best_loss[better] = (x + delta)[better], loss[better]
    return best_x.detach()

# Verifier

A *region* has a center `c` (a batch of images), a radius `r` and a norm.
For $\ell_\infty$, the region is the box $[c - r, c + r] \cap [0, 1]^{784}$.
For $\ell_2$, the region is the ball $\{x' : \|x' - c\|_2 \leq r\}$. We do not intersect the ball with $[0, 1]^{784}$.
This is sound, but the bounds are a little looser.

* `ibp_margin_bounds(model, c, r, norm, y)` is IBP from Problem 1. For $\ell_2$, the bounds of the first linear layer
  are exact: $W_i c + b_i \pm r \|W_i\|_2$ for each neuron $i$. The last linear layer is merged with the specification.
* `crown_margin_bounds(model, c, r, norm, y)` is CROWN. At the input, $\min_{x'} a^\top x'$ over the $\ell_2$ ball is
  $a^\top c - r \|a\|_2$.
* `certified(model, x, y, norm)` uses the larger of the two lower bounds.

The two bound functions return a tensor `m` with the shape `(batch, 10)`.
The value `m[:, j]` is a lower bound of $f_y - f_j$ on the region, and `m[:, y] = 0`. Do not change the verifier.

In [4]:
def first_layer_bounds(layer, c, r, norm):
    c = c.flatten(1)
    if norm == "linf":
        l, u = (c - r).clamp(0, 1), (c + r).clamp(0, 1)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
    else:
        mid, rad = layer(c), r * layer.weight.norm(dim=1)
    return mid - rad, mid + rad


def spec_matrix(y, n_classes=10):
    eye = torch.eye(n_classes, device=y.device)
    return eye[y].unsqueeze(1) - eye.unsqueeze(0)  # (batch, 10, 10): row j is e_y - e_j


def ibp_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    l, u = first_layer_bounds(layers[0], c, r, norm)
    for layer in layers[1:-1]:
        l, u = F.relu(l), F.relu(u)
        mid, rad = layer((u + l) / 2), ((u - l) / 2) @ layer.weight.abs().t()
        l, u = mid - rad, mid + rad
    l, u = F.relu(l), F.relu(u)
    last = layers[-1]
    w = last.weight[y].unsqueeze(1) - last.weight.unsqueeze(0)  # (batch, 10, n): W_y - W_j
    b = last.bias[y].unsqueeze(1) - last.bias.unsqueeze(0)
    mid, rad = (u + l) / 2, (u - l) / 2
    return (w @ mid.unsqueeze(2)).squeeze(2) - (w.abs() @ rad.unsqueeze(2)).squeeze(2) + b


def _relu_relaxation(l, u):
    active, unstable = l >= 0, (l < 0) & (u > 0)
    s = torch.where(active, torch.ones_like(l), torch.zeros_like(l))
    s = torch.where(unstable, u / (u - l).clamp(min=1e-12), s)
    c = torch.where(unstable, -s * l, torch.zeros_like(l))
    alpha = torch.where(active | (unstable & (u > -l)), torch.ones_like(l), torch.zeros_like(l))
    return alpha, s, c


def _backward_lower(A, layers, bounds, c0, r, norm):
    # Lower bound of A z, where z is the output of layers[-1] and bounds[i] are the bounds of the output of layers[i].
    const = torch.zeros(A.shape[:2], device=A.device)
    for i in range(len(layers) - 1, -1, -1):
        const = const + A @ layers[i].bias
        A = A @ layers[i].weight
        if i > 0:
            alpha, s, c = _relu_relaxation(*bounds[i - 1])
            a_pos, a_neg = A.clamp(min=0), A.clamp(max=0)
            const = const + (a_neg * c.unsqueeze(1)).sum(2)
            A = a_pos * alpha.unsqueeze(1) + a_neg * s.unsqueeze(1)
    c0 = c0.flatten(1)
    if norm == "linf":
        l0, u0 = (c0 - r).clamp(0, 1), (c0 + r).clamp(0, 1)
        return const + (A.clamp(min=0) * l0.unsqueeze(1)).sum(2) + (A.clamp(max=0) * u0.unsqueeze(1)).sum(2)
    return const + (A @ c0.unsqueeze(2)).squeeze(2) - r * A.norm(dim=2)


def crown_margin_bounds(model, c, r, norm, y):
    layers = linear_layers(model)
    bounds = []
    for k in range(len(layers) - 1):
        eye = torch.eye(layers[k].out_features, device=c.device).expand(len(c), -1, -1)
        lo = _backward_lower(eye, layers[:k + 1], bounds, c, r, norm)
        hi = -_backward_lower(-eye, layers[:k + 1], bounds, c, r, norm)
        bounds.append((lo, hi))
    return _backward_lower(spec_matrix(y), layers, bounds, c, r, norm)


@torch.no_grad()
def certified(model, x, y, norm, eps=None):
    eps = EPS[norm] if eps is None else eps
    m = torch.maximum(ibp_margin_bounds(model, x, eps, norm, y), crown_margin_bounds(model, x, eps, norm, y))
    m[torch.arange(len(y)), y] = float("inf")
    return m.min(dim=1).values > 0

# Evaluation

`evaluate(model, name)` uses the 10,000 test images. It returns the clean accuracy, the certified accuracy for each
norm and the union certified accuracy. With `attacks=True`, it also returns the PGD accuracy (50 steps) for each norm
and for the union. The PGD accuracy is an upper bound of the true robust accuracy.
Use `evaluate` for all the numbers in your report.

In [5]:
def evaluate(model, name, attacks=False, n=10000, batch_size=1000):
    model.eval()
    counts = {k: 0 for k in ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union"]}
    for x, y in batches(x_test[:n], y_test[:n], batch_size, False):
        with torch.no_grad():
            correct = model(x).argmax(1) == y
        cert = {p: certified(model, x, y, p) & correct for p in NORMS}
        counts["clean"] += correct.sum().item()
        counts["linf"] += cert["linf"].sum().item()
        counts["l2"] += cert["l2"].sum().item()
        counts["union"] += (cert["linf"] & cert["l2"]).sum().item()
        if attacks:
            rob = {}
            for p in NORMS:
                x_adv = pgd(model, x, y, EPS[p], p, steps=50)
                with torch.no_grad():
                    rob[p] = correct & (model(x_adv).argmax(1) == y)
                counts["pgd_" + p] += rob[p].sum().item()
            counts["pgd_union"] += (rob["linf"] & rob["l2"]).sum().item()
    res = {"name": name}
    res.update({k: 100 * v / n for k, v in counts.items() if attacks or not k.startswith("pgd")})
    return res


def print_results(rows):
    for r in rows:
        print(f"{r['name']:<26}" + "  ".join(f"{k} {v:6.2f}" for k, v in r.items()
                                     if k != "time" and isinstance(v, float)) +
              (f"  time/epoch {r['time']:5.1f} s" if "time" in r else ""))

# Training loop

For each batch, `train(model, loss_fn)` calls `loss_fn(model, x, y, t)` and does one optimizer step.
The value `t` is the training progress in epochs (a float). The schedule is the same as in Problem 1:

* For $t < 1$, `eps_scale(t)` is 0 (standard training).
* For $1 \leq t < 11$, `eps_scale(t)` increases linearly from 0 to 1.
* For $t \geq 11$, `eps_scale(t)` is 1.

A loss function uses the radius `eps_scale(t) * EPS[norm]`.

If you give the argument `gp_beta`, `train` uses `gradient_projection` (part (b)) when the radius is at its
final value ($t \geq 11$).

In [6]:
WARMUP, RAMP = 1, 10
LR = 4e-3


def eps_scale(t):
    return 0.0 if t <= WARMUP else min((t - WARMUP) / RAMP, 1.0)


def train(model, loss_fn, epochs=20, lr=LR, batch_size=512, seed=0, gp_beta=None, verbose=True, epoch_times=None):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[15], gamma=0.1)
    params = [p for p in model.parameters() if p.requires_grad]
    n_batches = (len(x_train) + batch_size - 1) // batch_size
    start = time.time()
    for epoch in range(epochs):
        epoch_start = time.perf_counter()
        model.train()
        total = 0.0
        for i, (x, y) in enumerate(batches(x_train, y_train, batch_size, True)):
            t = epoch + i / n_batches
            loss = loss_fn(model, x, y, t)
            opt.zero_grad()
            if gp_beta is not None and t >= WARMUP + RAMP:
                g_c = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, gn, gc in zip(params, g_n, g_c):
                    p.grad = gradient_projection(gn, gc, gp_beta)
            else:
                loss.backward()
            opt.step()
            total += loss.item()
        sched.step()
        if epoch_times is not None:
            epoch_times.append(time.perf_counter() - epoch_start)
        if verbose:
            print(f"epoch {epoch + 1:2d}  loss {total / n_batches:.4f}  time {time.time() - start:.0f} s")
    return (time.time() - start) / epochs


def standard_loss(model, x, y, t):
    return F.cross_entropy(model(x), y)

# (a) Single-norm certified training

CURE uses the SABR loss from the lectures for each norm. SABR finds an adversarial example $x^*$ with PGD, and then it
calculates the IBP loss on a small ball of radius $\tau = \lambda \epsilon$ around $x^*$. PGD searches for $x^*$ in the
ball of radius $\epsilon - \tau$ around $x$, so that the small ball is inside the full ball.

1. `propagation_region`: Find a center $x^*$ with `pgd` (5 steps) in the ball of radius $\epsilon - \tau$ around $x$.
   Attack the model in eval mode. Then set the model back to train mode.
2. `certified_loss`: Calculate the cross-entropy of $\hat{z} = -\underline{m}$. Here, $\underline{m}$ are the IBP lower
   bounds of the margins on the small ball. Thus $\hat{z}_y = 0$.
   With `reduction="none"`, return the loss of each image.
3. `single_norm_loss`: For $t < 1$, return the standard loss. After that, return `certified_loss` with the radius
   `eps_scale(t) * EPS[norm]`.

The default ratios are `LAM = {"linf": 0.7, "l2": 0.7}`.

In [7]:
LAM = {"linf": 0.7, "l2": 0.7}

def propagation_region(model, x, y, eps, lam, norm):
    tau = lam * eps
    if eps == tau:
        return x.detach(), tau
    was_training = model.training
    model.eval()
    try:
        center = pgd(model, x, y, eps - tau, norm, steps=5)
    finally:
        model.train(was_training)
    return center, tau

def small_ball_logits(model, x, y, eps, lam, norm):
    center, radius = propagation_region(model, x, y, eps, lam, norm)
    return -ibp_margin_bounds(model, center, radius, norm, y)

def certified_loss(model, x, y, eps, lam, norm, reduction="mean"):
    logits = small_ball_logits(model, x, y, eps, lam, norm)
    return F.cross_entropy(logits, y, reduction=reduction)

def single_norm_loss(model, x, y, t, norm):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    return certified_loss(model, x, y, eps_scale(t) * EPS[norm], LAM[norm], norm)

In [8]:
def run_method(name, loss_fn, seed, gp_beta=None, epochs=20):
    torch.manual_seed(seed)
    model = make_model()
    times = []
    sec = train(model, loss_fn, seed=seed, gp_beta=gp_beta, epochs=epochs,
                epoch_times=times)
    # Fix the evaluation attack stream independently of the training method.
    torch.manual_seed(10000 + seed)
    scores = evaluate(model, name, attacks=True)
    row = dict(scores, seed=seed, epochs=epochs, time=sec, epoch_seconds=times)
    out = Path("results")
    out.mkdir(exist_ok=True)
    target = out / f"{name}_seed{seed}.json"
    tmp = target.with_suffix(".tmp")
    tmp.write_text(json.dumps(row, indent=2) + chr(10))
    tmp.replace(target)
    print_results([dict(scores, time=sec)])
    return row

single_methods = {
    "standard": standard_loss,
    "linf": lambda m, x, y, t: single_norm_loss(m, x, y, t, "linf"),
    "l2": lambda m, x, y, t: single_norm_loss(m, x, y, t, "l2"),
}

In [ ]:
single_rows = [run_method(name, loss, 0) for name, loss in single_methods.items()]

Saved experiment results (each run has 20 epochs):
standard, seed 0: union 0.00%
linf, seed 0: union 5.16%
l2, seed 0: union 51.39%


# (b) CURE

Let $\hat{z}^\infty$ and $\hat{z}^2$ be the negative IBP margin bounds on the two small balls of an image.
These are the same values as in `certified_loss`. Let $d^p = \mathrm{softmax}(\hat{z}^p)$. Complete `cure_loss`:
$$\mathcal{L}_{CURE} = \frac{1}{n} \sum_{i=1}^{n} \max\left(\mathcal{L}_{CE}(\hat{z}^\infty_i, y_i),\ \mathcal{L}_{CE}(\hat{z}^2_i, y_i)\right)
+ \eta \cdot \frac{1}{|\gamma|} \sum_{i \in \gamma} \mathrm{KL}\left(d^2_i \,\|\, d^\infty_i\right),$$
Here, $\gamma$ is the set of images in the batch that are certified for $\ell_2$ on their small ball.
For these images, all entries of $\hat{z}^2_i$ except $\hat{z}^2_{i, y_i}$ are negative.
The first term is CURE-Max. The second term is the bound alignment (BA).
For $t < 1$, return the standard loss. If $\gamma$ is empty, the second term is 0.

Then complete `gradient_projection` (GP). Let $g_c$ be the gradient of `cure_loss`, and $g_n$ the gradient of the
cross-entropy loss on the clean images. For each parameter tensor:
$$g = \beta \cdot \mathrm{GP}(g_n, g_c) + (1 - \beta) \cdot g_c, \qquad
\mathrm{GP}(g_n, g_c) = \begin{cases} \cos(g_n, g_c) \cdot g_n & \text{if } \cos(g_n, g_c) > 0 \\ 0 & \text{otherwise.} \end{cases}$$
This is the same operation as in RAMP.

This is a simplified version of CURE. The paper is different in three ways:

* It propagates an $\ell_\infty$ box also for $\ell_2$.
* It uses a different $\ell_2$ attack.
* It applies GP to the updates of a full epoch.

In [10]:
def cure_loss(model, x, y, t, eta=0.5):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    z_inf, z_two = [small_ball_logits(model, x, y, eps_scale(t) * EPS[p], LAM[p], p)
                    for p in NORMS]
    per_norm = [F.cross_entropy(z, y, reduction="none") for z in (z_inf, z_two)]
    loss = torch.maximum(*per_norm).mean()
    if eta == 0:
        return loss
    competitors = z_two.masked_fill(F.one_hot(y, z_two.shape[1]).bool(), -torch.inf)
    certified_two = competitors.amax(1) < 0
    kl = F.kl_div(z_inf.log_softmax(1), z_two.log_softmax(1),
                  reduction="none", log_target=True).sum(1)
    return loss + eta * (kl * certified_two).sum() / certified_two.sum().clamp_min(1)

def gradient_projection(g_n, g_c, beta):
    cosine = F.cosine_similarity(g_n.reshape(1, -1), g_c.reshape(1, -1))
    return (1 - beta) * g_c + beta * cosine.clamp_min(0).squeeze() * g_n

cure_methods = {
    "cure_max": (lambda m, x, y, t: cure_loss(m, x, y, t, eta=0), None),
    "cure_ba": (cure_loss, None),
    "cure_ba_gp": (cure_loss, 0.5),
}

In [ ]:
cure_rows = [run_method(name, loss, seed, gp_beta=beta)
             for name, (loss, beta) in cure_methods.items() for seed in (0, 1)]

Saved experiment results (each run has 20 epochs):
cure_max, seed 0: union 57.06%
cure_max, seed 1: union 56.72%
cure_ba, seed 0: union 56.45%
cure_ba, seed 1: union 57.16%
cure_ba_gp, seed 0: union 56.08%
cure_ba_gp, seed 1: union 56.23%


# (c) Full-region, competitor-wise union loss

Use the full scheduled ball for each norm. For every rival class choose its larger upper bound on the rival-minus-label logit before computing cross-entropy. This covers different rivals exposed by different norms. The ablation uses the same full regions but takes a maximum after computing each norm's cross-entropy. Neither run uses PGD centers, bound alignment or gradient projection.

In [12]:
def full_region_loss(model, x, y, t, competitor_wise):
    if t < WARMUP:
        return standard_loss(model, x, y, t)
    z_inf, z_two = [-ibp_margin_bounds(model, x, eps_scale(t) * EPS[p], p, y)
                    for p in NORMS]
    if competitor_wise:
        return F.cross_entropy(torch.maximum(z_inf, z_two), y)
    losses = [F.cross_entropy(z, y, reduction="none") for z in (z_inf, z_two)]
    return torch.maximum(*losses).mean()

def my_loss(model, x, y, t):
    return full_region_loss(model, x, y, t, competitor_wise=True)

def ablation_loss(model, x, y, t):
    return full_region_loss(model, x, y, t, competitor_wise=False)

In [ ]:
proposal_rows = [run_method(name, loss, seed)
                 for name, loss in [("full_class", my_loss), ("full_image", ablation_loss)]
                 for seed in (0, 1)]

Saved experiment results (each run has 20 epochs):
full_class, seed 0: union 58.51%
full_class, seed 1: union 59.22%
full_image, seed 0: union 58.31%
full_image, seed 1: union 59.29%


In [14]:
def mean_rows(rows):
    grouped = {}
    keys = ["clean", "linf", "l2", "union", "pgd_linf", "pgd_l2", "pgd_union", "time"]
    for name in dict.fromkeys(row["name"] for row in rows):
        group = [row for row in rows if row["name"] == name]
        grouped[name] = {"name": name, **{k: float(np.mean([r[k] for r in group])) for k in keys}}
    return grouped

means = mean_rows(single_rows + cure_rows + proposal_rows)
print_results(means.values())
best = max(cure_methods, key=lambda name: means[name]["union"])
if means["full_class"]["time"] > means[best]["time"]:
    epochs = math.ceil(20 * means["full_class"]["time"] / means[best]["time"])
    loss, beta = cure_methods[best]
    matched = [run_method(f"{best}_matched", loss, seed, gp_beta=beta, epochs=epochs)
               for seed in (0, 1)]
    print_results(mean_rows(matched).values())

standard                  clean  97.14  linf   0.00  l2   0.00  union   0.00  pgd_linf   1.93  pgd_l2  11.72  pgd_union   1.92  time/epoch   0.1 s
linf                      clean  93.77  linf  81.03  l2   5.16  union   5.16  pgd_linf  84.67  pgd_l2  69.54  pgd_union  69.31  time/epoch   0.7 s
l2                        clean  92.42  linf  54.75  l2  58.79  union  51.39  pgd_linf  68.20  pgd_l2  78.10  pgd_union  67.78  time/epoch   1.3 s
cure_max                  clean  92.31  linf  67.48  l2  56.98  union  56.89  pgd_linf  76.85  pgd_l2  78.35  pgd_union  76.10  time/epoch   2.4 s
cure_ba                   clean  92.01  linf  67.41  l2  56.86  union  56.80  pgd_linf  76.58  pgd_l2  78.04  pgd_union  76.06  time/epoch   2.4 s
cure_ba_gp                clean  92.31  linf  66.95  l2  56.21  union  56.16  pgd_linf  76.12  pgd_l2  77.93  pgd_union  75.61  time/epoch   2.5 s
full_class                clean  88.91  linf  63.50  l2  59.27  union  58.86  pgd_linf  71.44  pgd_l2  74.05  pgd_unio